In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find Qwen2.5-14B-Instruct project root (paths.py). Run Jupyter with cwd project root or notebooks/."
    )
import paths


## Qwen-14B-Instruct 

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "Qwen/Qwen2.5-14B-Instruct"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(model_name)

prompt = "Give me a short introduction to large language model."
messages = [
    {"role": "system", "content": "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."},
    {"role": "user", "content": prompt}
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

generated_ids = model.generate(
    **model_inputs,
    max_new_tokens=2048,
    do_sample=False
)
generated_ids = [
    output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]


/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading checkpoint shards: 100%|███████████████████████████████| 8/8 [00:05<00:00,  1.50it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [2]:
import pandas as pd 
import re
import torch

# Load data
df = pd.read_csv(paths.DATA / "14B_High_Low_Irr_Data_Source_.csv")
print("Columns in dataset:")
print(df.columns.tolist())

sentence_df = pd.read_csv(paths.DATA / "2k_sentence_seperate.csv")
print("Columns in Snetence-Level Dataset:")
print(sentence_df.columns.tolist())

# Merge the dataframes based on Origin (from df) and ID (from sentence_df)
df = df.merge(sentence_df, left_on='Origin', right_on='ID', how='left')

print("\nColumns after merge:")
print(df.columns.tolist())

# Save the merged dataframe
df.to_csv(paths.DATA / "MedPAIR_1300_Sentence.csv", index=False)
print(f"\n✅ Merged dataset saved as paths.DATA / "MedPAIR_1300_Sentence.csv"")
print(f"📊 Total rows: {len(df)}")
print(f"📊 Total columns: {len(df.columns)}")

Columns in dataset:
['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High', 'question_options', 'answer_corr']
Columns in Snetence-Level Dataset:
['ID', 'centaur_question', 'sentence_number', 'answer', 'data_source', 'step1_excerpts', 'question_options', 'step1_sentences', 'sentence_1', 'sentence_2', 'sentence_3', 'sentence_4', 'sentence_5', 'sentence_6', 'sentence_7', 'sentence_8', 'sentence_9', 'sentence_10', 'sentence_11', 'sentence_12', 'sentence_13', 'sentence_14', 'sentence_15', 'sentence_16', 'sentence_17', 'sentence_18', 'sentence_19', 'sentence_20', 'sentence_21']

Columns after merge:
['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High', 'question_options_x', 'answer_corr', 'ID', 'centaur_question', 'sentence_number', 'answer', 'data_source', 'step1_excerpts', 'question_options_y', 'step1_sentences', 'sentence_1', 'sentence_2', 'sentence_3', 'sentence_4', 'sentence_5', 'sentence_6', 'sentence_7', 'sentence_8', 'sentence_9', 'sentence_10', 'sentence_1

In [7]:
import os
import re
import ast
import time
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
from time import time as timer

# === Prepare Output File ===
output_path = paths.DATA / "Qwen14B_annotated_MedPAIR_relevancy.csv"

# Create output columns
output_cols = df.columns.tolist() + ['Qwen14B_answer', 'Qwen14B_raw_response'] + [f"label_{i+1}" for i in range(21)]
sentence_cols = [f'sentence_{i}' for i in range(1, 22)]

# Initialize results list
results = []

# === Process Each Row ===
print(f"\nProcessing {len(df)} rows...")

for idx in range(len(df)):
    print(f"\n{'='*60}")
    print(f"Processing Row {idx + 1}/{len(df)}")
    print(f"{'='*60}")
    
    try:
        # === Extract Sentences and Options ===
        sentences = [df.loc[idx, col] for col in sentence_cols if pd.notna(df.loc[idx, col])]
        num_sentences = len(sentences)
        
        if num_sentences == 0:
            print(f"⚠️ No sentences found for row {idx}, skipping...")
            continue
        
        formatted_sentences = "\n".join(f"{j+1}. {s.strip()}" for j, s in enumerate(sentences))
        options = df.loc[idx, "question_options_x"]
        
        # === Display Memory Info ===
        if torch.cuda.is_available():
            free_mem = torch.cuda.memory_reserved(0) - torch.cuda.memory_allocated(0)
            print(f"🖥️ CUDA Free Memory: {free_mem / (1024 ** 2):.2f} MB")
        
        # === Format the Prompt ===
        prompt = f"""You are given a list of sentences from a clinical vignette and a multiple-choice clinical question.

Your task is twofold:
(1) Select the most appropriate answer from the given options.
(2) Label *each* of the {num_sentences} sentences as either [High Relevance], [Low Relevance], or [Irrelevant], based on its contribution to answering the question.

Definitions:
[High Relevance]: Sentences that directly support the correct answer with essential clinical information (e.g., diagnosis, key test results).
[Low Relevance]: Sentences that provide useful context or background but are not critical to answering.
[Irrelevant]: Sentences unrelated to the question or not useful for reasoning.

Question and Options:
{options}

Sentences:
{formatted_sentences}

Please provide your response in the following format:
Answer: [Your selected option letter]
Sentence 1: [High Relevance/Low Relevance/Irrelevant]
Sentence 2: [High Relevance/Low Relevance/Irrelevant]
...
(Continue for all {num_sentences} sentences)"""
        
        # === Generate Response Using Chat Template ===
        messages = [{"role": "user", "content": prompt}]
        
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
        print(f"🔍 Input Token Count: {len(model_inputs['input_ids'][0])}")
        
        # === Time the Generation ===
        start_time = timer()
        
        with torch.no_grad():
            generated_ids = model.generate(
                **model_inputs,
                max_new_tokens=1024,
                do_sample=False
            )
        
        end_time = timer()
        print(f"⏱️ Generation Time: {end_time - start_time:.2f} seconds")
        
        # === Decode Response ===
        generated_ids = [
            output_ids[len(input_ids):] 
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]
        response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]
        
        # === Display Raw Response ===
        print(f"\n{'='*60}")
        print("Raw Model Response:")
        print(f"{'='*60}")
        print(response)
        print(f"{'='*60}")
        
        # === Extract Answer ===
        answer_match = re.search(r"Answer:\s*([A-J])", response, re.IGNORECASE)
        extracted_answer = answer_match.group(1).upper() if answer_match else None
        
        # === Extract Sentence Labels ===
        labels = [None] * 21  # Initialize all labels as None
        
        for sent_num in range(1, num_sentences + 1):
            # Look for patterns like "Sentence 1: [High Relevance]" or "1. High Relevance"
            pattern = rf"Sentence\s*{sent_num}:\s*\[?(High Relevance|Low Relevance|Irrelevant)\]?"
            match = re.search(pattern, response, re.IGNORECASE)
            
            if match:
                labels[sent_num - 1] = match.group(1)
            else:
                # Try alternative pattern
                pattern2 = rf"{sent_num}\.\s*\[?(High Relevance|Low Relevance|Irrelevant)\]?"
                match2 = re.search(pattern2, response, re.IGNORECASE)
                if match2:
                    labels[sent_num - 1] = match2.group(1)
        
        print(f"\n📊 Extracted Answer: {extracted_answer}")
        print(f"📊 Extracted Labels: {labels[:num_sentences]}")
        
        # === Build Result Row ===
        result_row = df.loc[idx].tolist() + [extracted_answer, response] + labels
        results.append(result_row)
        
        # === Save Incrementally Every 10 Rows ===
        if (idx + 1) % 10 == 0:
            temp_df = pd.DataFrame(results, columns=output_cols)
            temp_df.to_csv(output_path, index=False)
            print(f"\n💾 Saved progress: {idx + 1} rows completed")
        
    except Exception as e:
        print(f"❌ Error at row {idx}: {e}")
        import traceback
        traceback.print_exc()
        # Add empty result to maintain row alignment
        result_row = df.loc[idx].tolist() + [None, str(e)] + [None] * 21
        results.append(result_row)
    
    # === Clear CUDA Cache ===
    torch.cuda.empty_cache()

# === Final Save ===
print("\n" + "="*60)
print("Processing Complete!")
print("="*60)

final_df = pd.DataFrame(results, columns=output_cols)
final_df.to_csv(output_path, index=False)
print(f"✅ Final results saved to: {output_path}")
print(f"📊 Total rows processed: {len(results)}")


Processing 1300 rows...

Processing Row 1/1300
🖥️ CUDA Free Memory: 107.29 MB
🔍 Input Token Count: 664
⏱️ Generation Time: 18.90 seconds

Raw Model Response:
Answer: D

Sentence 1: [Low Relevance] - Provides context about the patient's medical history and the duration of symptoms, but does not directly contribute to the next step in management.
Sentence 2: [Low Relevance] - Denies presence of other neurological symptoms, providing context but not directly relevant to the next step.
Sentence 3: [Low Relevance] - Normal findings in visual acuity, color vision, etc., provide context but do not directly inform the next step.
Sentence 4: [High Relevance] - Describes the ptosis and physical exam findings, indicating the need for further investigation into the cause of ptosis.
Sentence 5: [Low Relevance] - Absence of signs of inflammation does not directly guide the next step.
Sentence 6: [High Relevance] - Mild fullness and increased resistance suggest possible orbital pathology requiring f

⏱️ Generation Time: 14.37 seconds

Raw Model Response:
Answer: H

Sentence 1: [Low Relevance] - Provides context about the patient's age and the cause of injury, but does not contribute directly to the method of transporting the amputated finger.
Sentence 2: [Low Relevance] - Describes initial first aid measures taken by the teacher, which is not directly related to the transportation of the amputated finger.
Sentence 3: [Irrelevant] - Information about the patient's medical history and medications is not relevant to the method of transporting the amputated finger.
Sentence 4: [Irrelevant] - Vital signs do not impact the method of transporting the amputated finger.
Sentence 5: [Low Relevance] - Details about the wound are important for understanding the situation but are not directly relevant to the method of transporting the amputated finger.
Sentence 6: [Low Relevance] - Further details about the treatment provided in the emergency department are not necessary for determining the cor

⏱️ Generation Time: 21.36 seconds

Raw Model Response:
Answer: D

Sentence 1: [Low Relevance] - Provides context about the patient's ocular history but does not directly contribute to the next step in management.
Sentence 2: [High Relevance] - Indicates a non-healing conjunctival epithelial defect and underlying scleral thinning, suggesting a possible inflammatory or autoimmune condition.
Sentence 3: [Low Relevance] - Describes current treatment but does not influence the decision on the next step.
Sentence 4: [Low Relevance] - Lists medical history but does not directly impact the immediate management decision.
Sentence 5: [Low Relevance] - Provides additional medical history but is not crucial for deciding the next step.
Sentence 6: [High Relevance] - Severe decrease in visual acuity and low intraocular pressure in the affected eye suggest a serious condition requiring urgent intervention.
Sentence 7: [High Relevance] - Detailed findings from slitlamp examination indicate significant

⏱️ Generation Time: 12.44 seconds

Raw Model Response:
Answer: B. Acute lymphoblastic leukemia

Sentence 1: [Low Relevance] - This sentence provides basic demographic information but does not contribute directly to the diagnosis.
Sentence 2: [Low Relevance] - This sentence gives context about the patient's symptoms but does not contribute directly to the diagnosis.
Sentence 3: [Low Relevance] - This sentence provides additional context about the patient's health status but does not contribute directly to the diagnosis.
Sentence 4: [Irrelevant] - This sentence lists vital signs which do not contribute directly to the diagnosis.
Sentence 5: [Low Relevance] - This sentence mentions pallor, which could be related to anemia but does not directly point towards any specific type of leukemia.
Sentence 6: [High Relevance] - This sentence contains laboratory data showing a very high leukocyte count, which is indicative of acute leukemia, particularly acute lymphoblastic leukemia (ALL). The other

⏱️ Generation Time: 22.02 seconds

Raw Model Response:
Answer: A. Paranoid personality disorder

Sentence 1: [Low Relevance] - This sentence provides context about the patient's age and reason for visiting the physician, but it does not directly contribute to identifying the specific disorder.
Sentence 2: [Low Relevance] - This sentence gives background information about the patient's belief regarding his pantry door, which is not directly related to the disorder identification.
Sentence 3: [Low Relevance] - Similar to Sentence 2, this sentence provides context but does not directly contribute to the disorder identification.
Sentence 4: [High Relevance] - This sentence indicates a belief in being targeted or harmed by others, which is a characteristic symptom of paranoid personality disorder.
Sentence 5: [High Relevance] - This sentence further elaborates on the patient's delusional beliefs about his neighbors' intentions, supporting the diagnosis of paranoid personality disorder.
Sent

...

⏱️ Generation Time: 9.07 seconds

Raw Model Response:
Answer: B. Case series

Sentence 1: [Low Relevance] - This sentence provides context about the purpose of the study but does not directly inform us about the study design.
Sentence 2: [Low Relevance] - It gives information about the number of participants involved, which is useful but not crucial for identifying the study design.
Sentence 3: [Low Relevance] - This sentence describes the intervention but does not specify the study design.

The most relevant aspect here is that the study involves a group of patients receiving a specific treatment without comparing them to another group or control condition, which aligns with a case series design. However, none of the sentences explicitly state the study design, so they are labeled as low relevance rather than high relevance.

📊 Extracted Answer: B
📊 Extracted Labels: ['Low Relevance', 'Low Relevance', 'Low Relevance']

Processing Row 1292/1300
🖥️ CUDA Free Memory: 13.28 MB
🔍 Input Tok

⏱️ Generation Time: 22.10 seconds

Raw Model Response:
Answer: C. α1-Adrenergic antagonist

Explanation:
The patient's symptoms suggest an allergic reaction, possibly mild asthma exacerbation due to the bee sting. The presence of bilateral wheezing indicates airway constriction, which can be treated with medications that help relax bronchial smooth muscle. Among the options provided, α1-Adrenergic antagonists are not typically used for treating allergic reactions or asthma; however, the closest relevant class would be β2-adrenergic agonists, which are bronchodilators. But since the question asks for the most appropriate pharmacotherapy and considering the context of allergic reactions, α1-Adrenergic antagonists (such as phentolamine) might be considered in severe cases for their vasodilatory effects, though they are not first-line treatment. Given the options and the context, the closest logical choice would be an adrenergic antagonist that could potentially counteract the effects of c

❌ Error at row 1299: [Errno 122] Disk quota exceeded

Processing Complete!


Traceback (most recent call last):
  File "/tmp/ipykernel_2591533/4071521267.py", line 141, in <module>
    temp_df.to_csv(output_path, index=False)
  File "/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/pandas/util/_decorators.py", line 333, in wrapper
    return func(*args, **kwargs)
  File "/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/pandas/core/generic.py", line 3989, in to_csv
    return DataFrameRenderer(formatter).to_csv(
  File "/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/pandas/io/formats/format.py", line 1014, in to_csv
    csv_formatter.save()
  File "/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/pandas/io/formats/csvs.py", line 251, in save
    with get_handle(
  File "/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/pandas/io/common.py", line 157, in __exit__
    self.close()
  File "/home/yuexing/miniconda/envs/openai_env/lib/python3.10/site-packages/pandas/i

OSError: [Errno 122] Disk quota exceeded

## Concordance Rate Calculation

In [92]:
Qwen14B_Labels = pd.read_csv(paths.DATA / "Qwen14B_annotated_MedPAIR_relevancy.csv")
Qwen14B_Labels.columns

Index(['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High',
       'question_options_x', 'answer_corr', 'ID', 'centaur_question',
       'sentence_number', 'answer', 'data_source', 'step1_excerpts',
       'question_options_y', 'step1_sentences', 'sentence_1', 'sentence_2',
       'sentence_3', 'sentence_4', 'sentence_5', 'sentence_6', 'sentence_7',
       'sentence_8', 'sentence_9', 'sentence_10', 'sentence_11', 'sentence_12',
       'sentence_13', 'sentence_14', 'sentence_15', 'sentence_16',
       'sentence_17', 'sentence_18', 'sentence_19', 'sentence_20',
       'sentence_21', 'Qwen14B_answer', 'Qwen14B_raw_response', 'label_1',
       'label_2', 'label_3', 'label_4', 'label_5', 'label_6', 'label_7',
       'label_8', 'label_9', 'label_10', 'label_11', 'label_12', 'label_13',
       'label_14', 'label_15', 'label_16', 'label_17', 'label_18', 'label_19',
       'label_20', 'label_21'],
      dtype='object')

In [93]:
import pandas as pd

original_path = paths.DATA / "Centaur_Lab_First_Round_COMPLETE_RAW.csv"
trainee = pd.read_csv(original_path)
trainee.columns
# physician.loc[:, "q2":"q22"]

# Rename label_1–label_20 → q_1–q_20
rename_dict = {f"label_{i}": f"q{i}" for i in range(1, 21)}
Qwen14B_Labels = Qwen14B_Labels.rename(columns=rename_dict)

# Verify
Qwen14B_Labels.columns

Index(['Origin', 'data_source_df3', 'Patient_Profile', 'Low+Irr', 'High',
       'question_options_x', 'answer_corr', 'ID', 'centaur_question',
       'sentence_number', 'answer', 'data_source', 'step1_excerpts',
       'question_options_y', 'step1_sentences', 'sentence_1', 'sentence_2',
       'sentence_3', 'sentence_4', 'sentence_5', 'sentence_6', 'sentence_7',
       'sentence_8', 'sentence_9', 'sentence_10', 'sentence_11', 'sentence_12',
       'sentence_13', 'sentence_14', 'sentence_15', 'sentence_16',
       'sentence_17', 'sentence_18', 'sentence_19', 'sentence_20',
       'sentence_21', 'Qwen14B_answer', 'Qwen14B_raw_response', 'q1', 'q2',
       'q3', 'q4', 'q5', 'q6', 'q7', 'q8', 'q9', 'q10', 'q11', 'q12', 'q13',
       'q14', 'q15', 'q16', 'q17', 'q18', 'q19', 'q20', 'label_21'],
      dtype='object')

In [94]:
df = Qwen14B_Labels.copy()

# -------------------------
# 1. Empty rate calculation
# -------------------------
# Define empty as NaN or empty/whitespace-only string
is_empty = df['Qwen14B_answer'].isna() | (df['Qwen14B_answer'].astype(str).str.strip() == "")
empty_rate = is_empty.mean()

# -------------------------
# 2. Accuracy calculation
# -------------------------
# Compute accuracy only on non-empty predictions
valid_mask = ~is_empty

# Optional normalization for robust matching
pred = df.loc[valid_mask, 'Qwen14B_answer'].astype(str).str.strip().str.lower()
gt = df.loc[valid_mask, 'answer'].astype(str).str.strip().str.lower()

accuracy = (pred == gt).mean()

print(f"Qwen14B_answer empty rate: {empty_rate:.4f}")
print(f"Accuracy on non-empty predictions: {accuracy:.4f}")

Qwen14B_answer empty rate: 0.0629
Accuracy on non-empty predictions: 0.4316


In [95]:
cols_trainee = [f"q{i}" for i in range(1, 21)]

def transform_trainee(value):
    if pd.isna(value):
        return value
    
    val_str = str(value).strip().upper()
    
    # Check if numeric (including floats stored as strings)
    try:
        float(value)
        return "high relevance"
    except ValueError:
        pass
    
    if val_str == "REMOVED":
        return "low relevance"
    else:
        return value

trainee[cols_trainee] = trainee[cols_trainee].apply(lambda col: col.map(transform_trainee))

In [96]:
cols_qwen14B = [f"q{i}" for i in range(1, 21)]

def transform_Qwen14B(value):
    if pd.isna(value):
        return value
    val = str(value).strip().lower()
    if val == "not applicable":
        return ""
    elif val == "not relevant":
        return "low relevance"
    else:
        return value

Qwen14B_Labels[cols_qwen14B] = Qwen14B_Labels[cols_qwen14B].map(transform_Qwen14B)

In [97]:
trainee[["Origin", 'data_source_corr'] + cols_trainee].head(10)

,Origin,data_source_corr,q1,q2,q3,q4,q5,q6,q7,q8,...,q11,q12,q13,q14,q15,q16,q17,q18,q19,q20
0,ID0002,jama,high relevance,high relevance,low relevance,high relevance,low relevance,high relevance,low relevance,low relevance,...,high relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ID0003,medxpert,high relevance,high relevance,low relevance,high relevance,high relevance,high relevance,low relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ID0007,medbullets,low relevance,high relevance,high relevance,high relevance,high relevance,low relevance,high relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ID0009,jama,low relevance,high relevance,high relevance,high relevance,high relevance,high relevance,low relevance,low relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ID0010,medxpert,high relevance,low relevance,low relevance,low relevance,low relevance,low relevance,high relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ID0013,mmlu,high relevance,high relevance,high relevance,low relevance,low relevance,high relevance,high relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ID0015,jama,high relevance,high relevance,low relevance,high relevance,low relevance,high relevance,low relevance,low relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ID0016,jama,high relevance,low relevance,high relevance,low relevance,low relevance,low relevance,low relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ID0017,jama,high relevance,high relevance,high relevance,high relevance,low relevance,low relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ID0018,jama,high relevance,high relevance,low relevance,low relevance,high relevance,high relevance,high relevance,high relevance,...,high relevance,high relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [98]:
Qwen14B_Labels = Qwen14B_Labels.sort_values(by='Origin')
for c in cols_qwen14B:
    Qwen14B_Labels[c] = Qwen14B_Labels[c].apply(lambda x: x.lower() if isinstance(x, str) else x)
Qwen14B_Labels[["Origin"] + cols_qwen14B].head(10)

,Origin,q1,q2,q3,q4,q5,q6,q7,q8,q9,...,q11,q12,q13,q14,q15,q16,q17,q18,q19,q20
0,ID0002,low relevance,low relevance,low relevance,high relevance,low relevance,high relevance,low relevance,low relevance,low relevance,...,low relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ID0003,low relevance,low relevance,irrelevant,low relevance,low relevance,low relevance,low relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ID0007,low relevance,low relevance,low relevance,low relevance,low relevance,low relevance,low relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ID0009,high relevance,high relevance,high relevance,low relevance,high relevance,high relevance,low relevance,high relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ID0010,low relevance,low relevance,irrelevant,irrelevant,low relevance,low relevance,high relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ID0013,high relevance,low relevance,high relevance,low relevance,low relevance,low relevance,low relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ID0015,low relevance,low relevance,low relevance,high relevance,low relevance,high relevance,low relevance,low relevance,high relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ID0016,low relevance,low relevance,high relevance,low relevance,low relevance,low relevance,low relevance,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ID0017,low relevance,low relevance,low relevance,low relevance,low relevance,high relevance,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ID0018,low relevance,high relevance,low relevance,low relevance,low relevance,high relevance,high relevance,high relevance,high relevance,...,irrelevant,high relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [100]:
import numpy as np
import pandas as pd

# Columns to compare
cols_compare = [f"q{i}" for i in range(1, 20)]

# Create dictionary for fast lookup: Origin → trainee row
trainee_dict = {trainee.loc[i, "Origin"]: trainee.iloc[i] for i in range(len(trainee))}

def calculate_match(row_t, row_p):
    matches = 0
    total_valid = 0  # denominator: number of non-NaN labels

    for c in cols_compare:
        val_t = row_t[c]
        val_p = row_p[c]

        # Only consider this column if both values are not NaN
        if pd.notna(val_t) and pd.notna(val_p):
            total_valid += 1
            if val_t == val_p:
                matches += 1

    if total_valid == 0:
        return np.nan
    else:
        return f"{matches}/{total_valid}"  # numerator / denominator

# Apply only if Origins match
match_results = []
trainee_sources = []

for i in range(len(Qwen14B_Labels)):
    origin_p = Qwen14B_Labels.loc[i, "Origin"]
    if origin_p in trainee_dict:
        row_t = trainee_dict[origin_p]
        row_p = Qwen14B_Labels.iloc[i]
        match_results.append(calculate_match(row_t, row_p))
        trainee_sources.append(row_t.get("data_source_corr", np.nan))
    else:
        match_results.append(np.nan)
        trainee_sources.append(np.nan)

# Add results to DataFrame
Qwen14B_Labels["Match?"] = match_results
Qwen14B_Labels["data_source_corr_trainee"] = trainee_sources

# View results
Qwen14B_Labels[["Origin", "data_source_corr_trainee", "Match?"] + cols_compare].head(50)

,Origin,data_source_corr_trainee,Match?,q1,q2,q3,q4,q5,q6,q7,...,q10,q11,q12,q13,q14,q15,q16,q17,q18,q19
0,ID0002,jama,8/11,low relevance,low relevance,low relevance,high relevance,low relevance,high relevance,low relevance,...,low relevance,low relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,ID0003,medxpert,1/7,low relevance,low relevance,irrelevant,low relevance,low relevance,low relevance,low relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ID0007,medbullets,2/7,low relevance,low relevance,low relevance,low relevance,low relevance,low relevance,low relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,ID0009,jama,5/8,high relevance,high relevance,high relevance,low relevance,high relevance,high relevance,low relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,ID0010,medxpert,4/7,low relevance,low relevance,irrelevant,irrelevant,low relevance,low relevance,high relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,ID0013,mmlu,4/7,high relevance,low relevance,high relevance,low relevance,low relevance,low relevance,low relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,ID0015,jama,7/10,low relevance,low relevance,low relevance,high relevance,low relevance,high relevance,low relevance,...,low relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,ID0016,jama,6/7,low relevance,low relevance,high relevance,low relevance,low relevance,low relevance,low relevance,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,ID0017,jama,1/6,low relevance,low relevance,low relevance,low relevance,low relevance,high relevance,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,ID0018,jama,8/12,low relevance,high relevance,low relevance,low relevance,low relevance,high relevance,high relevance,...,irrelevant,irrelevant,high relevance,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [102]:
def combine_high_relevance_sentences(row):
    """
    Combines sentences where corresponding q columns have 'high relevance'
    """
    high_relevance_sentences = []
    
    # Iterate through q1 to q20
    for i in range(1, 21):
        q_col = f'q{i}'
        sentence_col = f'sentence_{i}'
        
        # Check if q column contains 'high relevance' (case-insensitive)
        if pd.notna(row.get(q_col)) and pd.notna(row.get(sentence_col)):
            if 'high relevance' in str(row[q_col]).lower():
                high_relevance_sentences.append(str(row[sentence_col]))
    
    # Combine all high relevance sentences with a separator
    if high_relevance_sentences:
        return ' '.join(high_relevance_sentences)
    else:
        return None  # or return '' for empty string

# Apply the function to create the new column
Qwen14B_Labels['Qwen14B_High_Relevance'] = Qwen14B_Labels.apply(combine_high_relevance_sentences, axis=1)

# Optional: Check the results
print(f"Total rows with high relevance sentences: {Qwen14B_Labels['Qwen14B_High_Relevance'].notna().sum()}")
print("\nSample of the new column:")
print(Qwen14B_Labels[['Qwen14B_High_Relevance']].head(20))

# Select columns to save (including Match?)
columns_to_save = list(Qwen14B_Labels.columns)
# Ensure 'Match?' is included if it exists
if 'Match?' in Qwen14B_Labels.columns:
    print("\n'Match?' column found and will be included in the CSV")
else:
    print("\nWarning: 'Match?' column not found in Qwen14B_Labels")

# Save to CSV with all columns (including Match?)
Qwen14B_Labels.to_csv(paths.DATA / "Qwen14B_annotated_MedPAIR_relevancy.csv", index=False)

Total rows with high relevance sentences: 1245

Sample of the new column:
                               Qwen14B_High_Relevance
0   4. External examination was notable for left u...
1                                                None
2                                                None
3   1. A woman in her 30s presented with multiple ...
4   7. The amputated finger was left in the classr...
5   1. A 3-year-old white girl is brought to the o...
6   4. At the time of the consultation, he was fre...
7   3. A slitlamp examination of the right eye rev...
8   6. Contrast-enhanced CT of the abdomen and pel...
9   2. After strabismus surgery, he was noted to h...
10  6. On examination, clear vesicles on erythemat...
11  4. An endometrial biopsy specimen shows regula...
12  5. After diagnosing a T6 extended, rotated lef...
13  1. A 47-year-old man comes to the physician 12...
14  6. Basic laboratory studies are ordered as see...
15  12. On physical exam, the patient has a flat a...
16  1. T

OSError: [Errno 122] Disk quota exceeded

In [89]:
import numpy as np
import pandas as pd
from scipy import stats

# Function to convert 'Match?' to numeric percentage
def match_to_percent(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, (int, float)):
        return float(x) * 100
    if isinstance(x, str) and "/" in x:
        try:
            num, denom = x.split("/")
            return (float(num) / float(denom)) * 100
        except:
            return np.nan
    return np.nan

# Create a numeric column for Match? in percentage
Qwen14B_Labels["Match_percent"] = Qwen14B_Labels["Match?"].apply(match_to_percent)

# Group by data_source_corr_trainee
grouped = Qwen14B_Labels.groupby("data_source_corr_trainee")["Match_percent"]

# Function to compute mean, SD, and 95% CI
def mean_sd_ci(series):
    series = series.dropna()
    n = len(series)
    if n == 0:
        return pd.Series({"mean": np.nan, "sd": np.nan, "CI_lower": np.nan, "CI_upper": np.nan, "n": 0})
    mean = series.mean()
    sd = series.std()  # standard deviation
    sem = stats.sem(series)  # standard error
    ci = sem * stats.t.ppf((1 + 0.95) / 2, n-1)  # 95% CI
    return pd.Series({"mean": mean, "sd": sd, "CI_lower": mean - ci, "CI_upper": mean + ci, "n": n})

# Apply function
summary = grouped.apply(mean_sd_ci).reset_index()

# Pivot from long to wide format
summary = summary.pivot(index='data_source_corr_trainee', columns='level_1', values='Match_percent').reset_index()

# Create beautiful formatted table
summary["Mean (%)"] = summary["mean"].apply(lambda x: f"{x:.2f}" if not pd.isna(x) else "N/A")
summary["SD (%)"] = summary["sd"].apply(lambda x: f"{x:.2f}" if not pd.isna(x) else "N/A")
summary["95% CI"] = summary.apply(
    lambda row: f"[{row['CI_lower']:.2f}, {row['CI_upper']:.2f}]" if not pd.isna(row['CI_lower']) else "N/A",
    axis=1
)
summary["N"] = summary["n"].astype(int)

# Select and rename columns for final display
result = summary[["data_source_corr_trainee", "N", "Mean (%)", "SD (%)", "95% CI"]]
result.columns = ["Data Source", "N", "Mean (%)", "SD (%)", "95% CI"]

result

,Data Source,N,Mean (%),SD (%),95% CI
0,jama,582,55.85,18.29,"[54.36, 57.34]"
1,medbullets,207,47.66,20.62,"[44.83, 50.48]"
2,medxpert,318,50.42,23.95,"[47.78, 53.06]"
3,mmlu,193,53.55,23.89,"[50.16, 56.94]"


In [91]:
import numpy as np
from scipy import stats

# Drop NaNs
all_matches = Qwen14B_Labels["Match_percent"].dropna()

n = len(all_matches)
if n == 0:
    overall_summary = {"N": 0, "Mean (%)": np.nan, "SD (%)": np.nan, "95% CI": (np.nan, np.nan)}
else:
    mean = all_matches.mean()
    sd = all_matches.std()
    sem = stats.sem(all_matches)
    ci = sem * stats.t.ppf((1 + 0.95) / 2, n-1)
    overall_summary = {
        "N": n,
        "Mean (%)": round(mean, 2),
        "SD (%)": round(sd, 2),
        "95% CI": [round(mean - ci, 2), round(mean + ci, 2)]
    }

overall_summary

{'N': 1300,
 'Mean (%)': np.float64(52.88),
 'SD (%)': np.float64(21.24),
 '95% CI': [np.float64(51.72), np.float64(54.03)]}